# Day 6 Demo 2 - The same ladder, **with Strands**

Strands writes the agent loop for you. Same six stages as Demo 1: call -> workflow -> agent -> agent+tools -> multi-agent -> multi-agent+tools.

> **Strands calls real Bedrock under the hood - there is no offline mode here.** Run this with AWS credentials and model access (see setup). Compare the line counts with Demo 1.

## Before you run (console + setup, in this order)

1. **Enable model access** - Bedrock console -> **Model catalog** -> **Anthropic Claude Sonnet 4.5** in **us-west-2**.
2. **Install** - `pip install strands-agents`.
3. **Credentials** - `aws configure` (or env / role). Verify with `aws sts get-caller-identity`.
4. **Set the placeholders** - `REGION`, `MODEL` (a `us.` inference profile).

In [ ]:
# %pip install strands-agents
from strands import Agent, tool
from strands.models import BedrockModel

REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # <-- a us. inference profile you have ENABLED

bedrock = BedrockModel(model_id=MODEL, region_name=REGION, temperature=0.2)
print("Strands model ready:", MODEL)

## 1 - A single LLM call

An `Agent` with no tools is just an LLM. Call it like a function.

In [ ]:
llm = Agent(model=bedrock)
print(llm("In one sentence, what is an inference profile in Amazon Bedrock?"))

**Best practices**
- Set model/temperature on `BedrockModel`, not per call.
- `print(agent(...))` shows the text; the return value is an `AgentResult` (use `str(result)` or `result.message`).

**Alternatives**
- Raw `converse` (Demo 1) when you need full control of the request payload or custom retries.

## 2 - An LLM workflow (a fixed pipeline)

Two single-purpose agents, called in an order you decide.

In [ ]:
extractor = Agent(model=bedrock,
                  system_prompt='Extract PNR and intent. Reply with ONLY JSON: {"pnr": "...", "intent": "..."}.')
writer    = Agent(model=bedrock,
                  system_prompt="Write a warm 2-sentence customer reply from the given details.")

def workflow(customer_msg):
    data  = extractor(customer_msg)
    reply = writer(str(data))
    return data, reply

ex, reply = workflow("Hi, my flight on PNR JX48Q2 got cancelled. What now?")
print("extracted:", ex)
print("reply    :", reply)

**Best practices**
- One agent per step = clear, testable units.
- Validate structured output between steps (`json.loads`).

**Alternatives**
- A single agent for trivial chains.
- `strands.multiagent.GraphBuilder` for a declarative pipeline with branches.

## 3 - An *agent* (stateful, multi-turn) - no tools

A Strands `Agent` keeps **conversation memory** across calls. With no tools the loop is a single turn, but the agent now has *state* - the thing a bare call lacks.

In [ ]:
chat = Agent(model=bedrock, system_prompt="You are TravelMind support. Be concise.")
print(chat("My PNR is JX48Q2 and my flight was cancelled."))
print("---")
print(chat("Remind me - what was my PNR?"))   # the agent remembers from the previous turn

**Best practices**
- Conversation history is automatic and grows each call - it costs input tokens; create a new `Agent` to reset.
- Use a `conversation_manager` to cap/trim history for long sessions.

**Alternatives**
- Raw `converse` (Demo 1) where you manage the `messages` list yourself.
- The loop does real work once the agent has **tools** (next).

## 4 - An agent **with tool calls**

`@tool` turns a function into a tool: the **docstring** is the description, the **type hints** are the schema. Strands runs the whole tool-use loop you hand-wrote in Demo 1.

In [ ]:
@tool
def lookup_booking(pnr: str) -> dict:
    """Look up a booking by its PNR code."""
    return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}

@tool
def get_disruption_reason(pnr: str) -> dict:
    """Why a booking was delayed or cancelled."""
    return {"pnr": pnr, "reason": "weather", "detail": "Heavy fog at origin"}

@tool
def get_rebooking_options(pnr: str) -> list:
    """Alternative flights for a disrupted booking."""
    return [{"flight": "AI-318", "dep": "18:40"}, {"flight": "6E-552", "dep": "21:15"}]

travel_agent = Agent(model=bedrock,
                     tools=[lookup_booking, get_disruption_reason, get_rebooking_options],
                     system_prompt="You are TravelMind, a booking-exception assistant. Never invent a PNR.")

print(travel_agent("My flight on JX48Q2 was disrupted - what happened and what are my options?"))

**Best practices**
- Write a real docstring and precise type hints - they ARE the tool contract the model reads.
- Compare with Demo 1 stage 4: no manual `toolUseId`, no append step, no loop - Strands handles all of it.

**Alternatives**
- The hand-rolled loop (Demo 1) when you need to inspect or customise every turn.
- A `bedrock-agent-runtime` Knowledge-Base retrieve can be wrapped as a `@tool` too.

## 5 - *Multi-agent* - the "agents as tools" pattern

Wrap each specialist `Agent` in a `@tool`, then give those tools to an **orchestrator** agent. The orchestrator decides which specialist to call.

In [ ]:
billing_agent    = Agent(model=bedrock, system_prompt="You are a billing specialist. Be precise about refunds and fees.")
disruption_agent = Agent(model=bedrock, system_prompt="You are a flight-disruption specialist. Be specific and practical.")

@tool
def handle_billing(question: str) -> str:
    """Answer billing, refund, and fee questions."""
    return str(billing_agent(question))

@tool
def handle_disruption(question: str) -> str:
    """Answer flight delay or cancellation questions."""
    return str(disruption_agent(question))

orchestrator = Agent(model=bedrock,
                     tools=[handle_billing, handle_disruption],
                     system_prompt="Route each customer query to the right specialist tool, then return a warm reply under 60 words.")

print(orchestrator("My flight was cancelled and I want to know my options."))

**Best practices**
- The orchestrator's system prompt should state *when* to use each specialist tool.
- Keep specialists single-domain; the orchestrator owns coordination.

**Alternatives - Swarm (autonomous handoff). Each agent needs a UNIQUE `name`:**
```python
from strands.multiagent import Swarm
triage = Agent(model=bedrock, name="triage", system_prompt="Route and hand off to a specialist.")
disr   = Agent(model=bedrock, name="disruption", system_prompt="Flight-disruption specialist.")
swarm  = Swarm([triage, disr])
print(swarm("My flight was cancelled, what now?"))
```
Also `strands.multiagent.GraphBuilder` for an explicit DAG of agents.

## 6 - Multi-agent **with tool calls**

Now a specialist itself uses tools. The disruption specialist gets the booking tools; the orchestrator coordinates.

In [ ]:
disruption_with_tools = Agent(
    model=bedrock,
    tools=[lookup_booking, get_disruption_reason, get_rebooking_options],
    system_prompt="You are a flight-disruption specialist. Use the tools to look up the booking before answering.")

@tool
def handle_disruption_v2(question: str) -> str:
    """Resolve a flight disruption end to end (looks up the booking, the reason, and the options)."""
    return str(disruption_with_tools(question))

orchestrator2 = Agent(
    model=bedrock,
    tools=[handle_billing, handle_disruption_v2],
    system_prompt="Route each query to the right specialist tool, then return a warm reply under 60 words.")

print(orchestrator2("Flight on JX48Q2 cancelled - what happened and what can I do?"))

**Best practices**
- A tool-using specialist wrapped as a tool = nested loops; log and bound them.
- Give each specialist only the tools its domain needs.

**Alternatives**
- `Swarm` / `GraphBuilder` where each node is itself a tool-using agent.
- Flatten to one agent with all tools when the coordination overhead is not worth it.

## Recap

Same ladder as Demo 1, a fraction of the code. Strands gives you the loop, the tool wiring, and conversation state; you focus on the tools and the roles. Next, **Demo 3** hosts one of these agents on **AgentCore**.